# L11 · DPO: An RL Objective Without Online RL

## Goal

- compute chosen/rejected log-ratios
- interpret beta
- explain the difference from online RL

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L11:toy:42").hexdigest()
print(f"lesson=L11 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L11 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:34e94f892bcb390a2847aa160775bea8eeb0e49ffef2a64e7d2a6f7fe35c6cdc data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: preference data → **DPO** → offline-alignment evaluation

$$L_{DPO}=-\log\sigma\left(\beta\left[(\log\pi_\theta(y_w|x)-\log\pi_{ref}(y_w|x))-(\log\pi_\theta(y_l|x)-\log\pi_{ref}(y_l|x))\right]\right)$$

DPO applies a logistic loss to how much more the chosen response improves over the rejected one relative to a reference. It removes a separate reward model and online rollout, but reference policy and preference data still define an implicit RL problem.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** Does the loss stay the same when chosen and rejected are swapped? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>Generally no. The preference margin changes sign and evaluates the opposite side of the sigmoid.</details>

In [2]:
from rl_study.algorithms.dpo import dpo_loss
policy_chosen = torch.tensor([-1.0, -0.5])
policy_rejected = torch.tensor([-2.0, -0.4])
reference_chosen = torch.tensor([-1.4, -0.7])
reference_rejected = torch.tensor([-1.8, -0.6])
dpo_output = dpo_loss(
    policy_chosen, policy_rejected, reference_chosen, reference_rejected, beta=0.2
)
swapped_output = dpo_loss(
    policy_rejected, policy_chosen, reference_rejected, reference_chosen, beta=0.2
)
print({"logits": dpo_output.logits.tolist(),
       "loss": round(float(dpo_output.loss), 4),
       "swapped_loss": round(float(swapped_output.loss), 4)})

{'logits': [0.12000000476837158, -5.9604645663569045e-09], 'loss': 0.664, 'swapped_loss': 0.724}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Summed and mean sequence log-probabilities have different length bias, so reduction must be explicit. IPO and KTO are alternatives with different assumptions about preference noise and data shape.

**Common trap:** The loss remains finite if the reference term is dropped or pair order is reversed. Hand-calculation parity and a swapped-pair test catch semantic errors. Regression tests: `test_dpo_loss_matches_hand_calculation`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert dpo_output.loss.item() != swapped_output.loss.item()
assert torch.isfinite(dpo_output.loss)
print("checks=passed")

checks=passed


**Recall:** What anchor disappears if the reference model is removed from DPO? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** The original-pair loss 0.664 differs from swapped loss 0.724. One printed margin is near zero, so that sample carries a weak preference signal.
- Executable checks: `test_dpo_loss_matches_hand_calculation`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L12 returns to online rollouts and normalizes several rewards within each prompt group.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

[Implementation note](../../docs/algorithms/dpo.md) · [Course map](../../docs/course-map.en.md)

## Sources

- `dpo-2023` — `docs/sources.yml`
- `repo-dpo` — `docs/sources.yml`